#### NOTE: For creating the GIFs from this notebook, it should be run from `jupyter-lab`, since it (most likely) will not work from VSCode.

In [68]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

import pandas as pd
import numpy as np
from rdkit.Chem import MolFromMolFile, SDWriter, MolToPDBFile

In [6]:
fixer = PDBFixer(filename="../../../examples/qmmm/biocomplex/3GPH.pdb")
fixer.removeChains([i for i,c in enumerate(fixer.topology.chains()) if c.id != "A"])
PDBFile.writeFile(fixer.topology, fixer.positions, open("../../../examples/qmmm/biocomplex/3GPH_chainA.pdb","w"))

In [7]:
fixer = PDBFixer(filename="../../../examples/qmmm/biocomplex/3GPH_chainA.pdb")
fixer.findMissingResidues()         # safe on RCSB version
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.0)

PDBFile.writeFile(fixer.topology, fixer.positions, open("../../../examples/qmmm/biocomplex/3GPH_chainA_fixed.pdb","w"))

In [8]:
# --- 1) read pocket center from P2Rank csv ---
df = pd.read_csv("../../../examples/qmmm/biocomplex/3GPH_chainA_fixed.pdb_predictions.csv")
df.keys()

# fix keys by removing leading/trailing spaces
df.columns = [key.strip() for key in df.columns]
row = df.iloc[0]
row["name"]

'pocket1  '

In [9]:
row = df.iloc[0]
cx, cy, cz = float(row["center_x"]), float(row["center_y"]), float(row["center_z"])
pocket_center = np.array([cx, cy, cz])

# --- 2) load ligand and translate its centroid to pocket center ---
lig = MolFromMolFile("../../../examples/qmmm/biocomplex/C2H6S2.sdf", removeHs=False)
conf = lig.GetConformer()

coords = np.array([[conf.GetAtomPosition(i).x,
                    conf.GetAtomPosition(i).y,
                    conf.GetAtomPosition(i).z]
                   for i in range(lig.GetNumAtoms())])
centroid = coords.mean(axis=0)
shift = pocket_center - centroid

for i in range(lig.GetNumAtoms()):
    p = conf.GetAtomPosition(i)
    conf.SetAtomPosition(i, (p.x + shift[0], p.y + shift[1], p.z + shift[2]))

# save translated ligand
SDWriter("../../../examples/qmmm/biocomplex/C2H6S2_in_pocket1.sdf").write(lig)
MolToPDBFile(lig, "../../../examples/qmmm/biocomplex/C2H6S2_in_pocket1.pdb")

In [10]:
from openmm import app, unit
import openmm as mm

# protein with correct residues, chain ids, TER, hydrogens
prot = app.PDBFile("../../../examples/qmmm/biocomplex/3GPH_chainA_fixed.pdb")

# ligand (pure xyz positions, new residue UNL)
lig  = app.PDBFile("../../../examples/qmmm/biocomplex/C2H6S2_in_pocket1.pdb")

mod = app.Modeller(prot.topology, prot.positions)

# Add ligand to protein
mod.add(lig.topology, lig.positions)

app.PDBFile.writeFile(mod.topology, mod.positions,
                      open("../../../examples/qmmm/biocomplex/3GPH_chainA_ligand_merged.pdb", "w"))

In [11]:
from openff.toolkit import Molecule
from openmmforcefields.generators import SMIRNOFFTemplateGenerator

complex_pdb_path = "../../../examples/qmmm/biocomplex/3GPH_chainA_ligand_merged.pdb"
lig_sdf_path     = "../../../examples/qmmm/biocomplex/C2H6S2.sdf"

LIGAND_RESNAME = "UNL"

# 1) Load PDB
pdb = app.PDBFile(complex_pdb_path)

# 2) Strip non-standard residues except ligand, waters, ions
standard_aas = {
    "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS",
    "ILE","LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"
}
keep_resnames = standard_aas | {LIGAND_RESNAME, "HOH", "WAT", "NA", "CL"}

modeller = app.Modeller(pdb.topology, pdb.positions)
to_delete = [res for res in modeller.topology.residues()
             if res.name not in keep_resnames]
modeller.delete(to_delete)

# 3) Protein / water FF
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")

# 4) Ligand via OpenFF, assign charges with RDKit (Gasteiger)
lig_mol = Molecule.from_file(lig_sdf_path)
lig_mol.assign_partial_charges("gasteiger")  # uses RDKit, which you have

# SMIRNOFF generator – no extra kwargs; it will use user charges on lig_mol
smirnoff_gen = SMIRNOFFTemplateGenerator(
    molecules=[lig_mol],
    forcefield="openff_unconstrained-2.1.0.offxml",  # or "openff-2.2.1" etc.
)
forcefield.registerTemplateGenerator(smirnoff_gen.generator)

# 5) Add hydrogens to protein/waters
modeller.addHydrogens(forcefield, pH=7.0)

# 6) Build system
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=app.HBonds,
    ewaldErrorTolerance=1.0e-4,
    ignoreExternalBonds=True,
)

positions = modeller.positions
print("System built! #particles:", system.getNumParticles())

System built! #particles: 7594


In [12]:
def show_modeller(modeller):
    import io, nglview as nv
    buf = io.StringIO()
    app.PDBFile.writeFile(modeller.topology, modeller.positions, buf, keepIds=True)
    buf.seek(0)
    v = nv.show_text(buf.getvalue())
    v.clear()
    v.add_cartoon("protein")
    v.add_ball_and_stick("UNL")
    return v

show_modeller(modeller)

/Users/luismantilla/Desktop/UofT/NVIDIA/trovolut/.venv/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


NGLWidget()

In [16]:
from openmmtools.alchemy import AbsoluteAlchemicalFactory, AlchemicalRegion
from pymbar import MBAR, timeseries

# ---- 1) Define alchemical region (ligand atoms) ----
ligand_atoms = [atom.index for atom in modeller.topology.atoms()
                if atom.residue.name == LIGAND_RESNAME]

alch_region  = AlchemicalRegion(
    alchemical_atoms=ligand_atoms,
    annihilate_electrostatics=True,
    annihilate_sterics=True,
)

factory      = AbsoluteAlchemicalFactory(
    consistent_exceptions=False,
    disable_alchemical_dispersion_correction=True,
)

alch_system  = factory.create_alchemical_system(system, alch_region)

# ---- 2) Simulation setup ----
temperature = 300 * unit.kelvin
friction    = 1.0 / unit.picosecond
timestep    = 1.0 * unit.femtoseconds

integrator  = mm.LangevinIntegrator(temperature, friction, timestep)

# pick your platform here
platform = mm.Platform.getPlatformByName("CPU")  # or "CPU"

simulation = app.Simulation(modeller.topology, alch_system, integrator, platform)
simulation.context.setPositions(positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

# separate context for energy evaluations
energy_integrator = mm.VerletIntegrator(timestep)
energy_context    = mm.Context(alch_system, energy_integrator, platform)


Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [17]:
# ---- 3) Lambda schedule ----
lambdas = np.linspace(0.0, 1.0, 6)  # 0..1, 6 windows
nstates = len(lambdas)

# MD / sampling params (toy values; bump for real calc)
n_equil_steps   = 500
n_steps_between = 100
n_samples       = 20   # per state

# storage for reduced potentials u_kln (k: sim state, l: eval state, n: sample)
u_kln = np.zeros((nstates, nstates, n_samples), dtype=np.float64)

# helper: set lambda_* on a context
def set_lambda_on_context(context, lam):
    for pname in ["lambda_sterics", "lambda_electrostatics",
                  "lambda_torsions", "lambda_angles"]:
        try:
            context.setParameter(pname, lam)
        except Exception:
            pass

# precompute beta
kT   = (unit.MOLAR_GAS_CONSTANT_R * temperature).value_in_unit(unit.kilojoule_per_mole)
beta = 1.0 / kT   # beta is now a dimensionless float

# ---- 4) Loop over λ, run MD, collect energies ----
for k, lam in enumerate(lambdas):
    print(f"State {k} / {nstates-1}, λ = {lam:.3f}")
    set_lambda_on_context(simulation.context, lam)
    simulation.minimizeEnergy(maxIterations=200)
    simulation.context.setVelocitiesToTemperature(temperature)

    # equilibration
    simulation.step(n_equil_steps)

    for n in range(n_samples):
        simulation.step(n_steps_between)
        state = simulation.context.getState(getPositions=True, getEnergy=True)
        pos   = state.getPositions(asNumpy=True)

        # copy positions into energy context once per sample
        energy_context.setPositions(pos)

        # evaluate U at all lambda states
        for l, lam_eval in enumerate(lambdas):
            set_lambda_on_context(energy_context, lam_eval)
            e = energy_context.getState(getEnergy=True).getPotentialEnergy()
            u_kln[k, l, n] = beta * e.value_in_unit(unit.kilojoule_per_mole)


State 0 / 5, λ = 0.000
State 1 / 5, λ = 0.200
State 2 / 5, λ = 0.400
State 3 / 5, λ = 0.600
State 4 / 5, λ = 0.800
State 5 / 5, λ = 1.000


In [18]:
# Save trajectory frames for GIF visualization
import os
from openmm.app import PDBFile
import imageio
from IPython.display import Image
import tempfile


# Storage for trajectories per lambda state
trajectories = {k: [] for k in range(nstates)}

# Re-run shorter simulation to collect frames for visualization
n_frames = 50  # frames per lambda for 5-sec GIF at 10fps
steps_per_frame = 50

print("Collecting trajectory frames for GIF visualization...")
for k, lam in enumerate(lambdas):
    print(f"  λ = {lam:.3f}")
    set_lambda_on_context(simulation.context, lam)
    simulation.minimizeEnergy(maxIterations=100)
    simulation.context.setVelocitiesToTemperature(temperature)
    simulation.step(200)  # brief equilibration
    
    for frame in range(n_frames):
        simulation.step(steps_per_frame)
        state = simulation.context.getState(getPositions=True)
        trajectories[k].append(state.getPositions(asNumpy=True))

print("Done collecting frames.")

  λ = 0.000
  λ = 0.200
  λ = 0.400
  λ = 0.600
  λ = 0.800
  λ = 1.000
Done collecting frames.


In [19]:
# Generate GIFs with lambda-dependent ligand transparency using NGLView + MDTraj
import io
import nglview as nv
import mdtraj as md
from PIL import Image as PILImage
import time

os.makedirs("lambda_gifs", exist_ok=True)

def write_multimodel_pdb(topology, positions_list, filename):
    """Write trajectory as multi-model PDB for NGLView."""
    with open(filename, 'w') as f:
        for i, pos in enumerate(positions_list):
            f.write(f"MODEL     {i+1}\n")
            buf = io.StringIO()
            PDBFile.writeFile(topology, pos, buf)
            # Skip header lines, keep ATOM/HETATM/TER
            for line in buf.getvalue().split('\n'):
                if line.startswith(('ATOM', 'HETATM', 'TER')):
                    f.write(line + '\n')
            f.write("ENDMDL\n")
        f.write("END\n")

# Create views for each lambda state
print("Creating NGLView trajectory widgets for each λ state...")
print("Run each cell below to see the animation")
print()

views = {}
trajs = {}  # Store MDTraj trajectories
for k, lam in enumerate(lambdas):
    pdb_file = f"lambda_gifs/traj_lambda_{lam:.2f}.pdb"
    write_multimodel_pdb(modeller.topology, trajectories[k], pdb_file)
    alpha = max(0.15, lam)
    
    # Load with MDTraj to get proper trajectory
    traj = md.load(pdb_file)
    trajs[k] = traj
    views[k] = (lam, alpha, pdb_file, traj)
    print(f"  λ = {lam:.2f} → {pdb_file} (opacity={alpha:.2f}, frames={traj.n_frames})")

Creating NGLView trajectory widgets for each λ state...
Run each cell below to see the animation

  λ = 0.00 → lambda_gifs/traj_lambda_0.00.pdb (opacity=0.15, frames=50)
  λ = 0.20 → lambda_gifs/traj_lambda_0.20.pdb (opacity=0.20, frames=50)
  λ = 0.40 → lambda_gifs/traj_lambda_0.40.pdb (opacity=0.40, frames=50)
  λ = 0.60 → lambda_gifs/traj_lambda_0.60.pdb (opacity=0.60, frames=50)
  λ = 0.80 → lambda_gifs/traj_lambda_0.80.pdb (opacity=0.80, frames=50)
  λ = 1.00 → lambda_gifs/traj_lambda_1.00.pdb (opacity=1.00, frames=50)


In [86]:
lam, alpha, pdb_file, traj = views[0]
view0 = nv.show_mdtraj(traj)
view0.clear()
view0.add_cartoon("protein")
view0.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view0.camera = "orthographic"
view0.background = "white"
view0.center("UNL")

size = 500
view0.layout.width  = f"{size}px"
view0.layout.height = f"{size}px"

view0.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)

print(f"λ = {lam:.2f} (decoupled - ghost ligand, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames} - Use slider below to navigate")
view0

λ = 0.00 (decoupled - ghost ligand, opacity=0.15)
Frames: 50 - Use slider below to navigate


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [87]:
lam, alpha, pdb_file, traj = views[1]
view1 = nv.show_mdtraj(traj)
view1.clear()
view1.add_cartoon("protein")
view1.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view1.camera = "orthographic"
view1.background = "white"
view1.center("UNL")

size = 500
view1.layout.width  = f"{size}px"
view1.layout.height = f"{size}px"

view1.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)


print(f"λ = {lam:.2f} (partially coupled, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames}")
view1

λ = 0.20 (partially coupled, opacity=0.20)
Frames: 50


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [88]:
lam, alpha, pdb_file, traj = views[2]
view2 = nv.show_mdtraj(traj)
view2.clear()
view2.add_cartoon("protein")
view2.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view2.camera = "orthographic"
view2.background = "white"
view2.center("UNL")

size = 500
view2.layout.width  = f"{size}px"
view2.layout.height = f"{size}px"

view2.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)


print(f"λ = {lam:.2f} (mostly coupled, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames}")
view2

λ = 0.40 (mostly coupled, opacity=0.40)
Frames: 50


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [89]:
lam, alpha, pdb_file, traj = views[3]
view3 = nv.show_mdtraj(traj)
view3.clear()
view3.add_cartoon("protein")
view3.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view3.camera = "orthographic"
view3.background = "white"
view3.center("UNL")

size = 500
view3.layout.width  = f"{size}px"
view3.layout.height = f"{size}px"

view3.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)


print(f"λ = {lam:.2f} (fully coupled - solid ligand, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames}")
view3

λ = 0.60 (fully coupled - solid ligand, opacity=0.60)
Frames: 50


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [90]:
lam, alpha, pdb_file, traj = views[4]
view4 = nv.show_mdtraj(traj)
view4.clear()
view4.add_cartoon("protein")
view4.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view4.camera = "orthographic"
view4.background = "white"
view4.center("UNL")

size = 500
view4.layout.width  = f"{size}px"
view4.layout.height = f"{size}px"

view4.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)


print(f"λ = {lam:.2f} (decoupled - ghost ligand, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames} - Use slider below to navigate")
view4

λ = 0.80 (decoupled - ghost ligand, opacity=0.80)
Frames: 50 - Use slider below to navigate


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [91]:
lam, alpha, pdb_file, traj = views[5]
view5 = nv.show_mdtraj(traj)
view5.clear()
view5.add_cartoon("protein")
view5.add_ball_and_stick("UNL", opacity=alpha, color="orange")
view5.camera = "orthographic"
view5.background = "white"
view5.center("UNL")

size = 500
view5.layout.width  = f"{size}px"
view5.layout.height = f"{size}px"

view5.add_ball_and_stick("resname HOH or resname WAT or resname SOL", color="skyblue", opacity=0.05)


print(f"λ = {lam:.2f} (decoupled - ghost ligand, opacity={alpha:.2f})")
print(f"Frames: {traj.n_frames} - Use slider below to navigate")
view5

λ = 1.00 (decoupled - ghost ligand, opacity=1.00)
Frames: 50 - Use slider below to navigate


NGLWidget(layout=Layout(height='500px', width='500px'), max_frame=49)

In [93]:
import os, time
from time import sleep
import ipywidgets as widgets
from IPython.display import display, clear_output

all_views = [view0, view1, view2, view3, view4, view5]

N_FRAMES = 50
OUT = widgets.Output()
display(OUT)

def activate_view(view, size=450):
    view.layout.width = f"{size}px"
    view.layout.height = f"{size}px"
    with OUT:
        clear_output(wait=True)
        display(view)
    sleep(1.5)  # let WebGL mount + render

for lam, view in zip(lambdas, all_views):
    activate_view(view)

    tag = f"lambda_{int(round(lam*100)):03d}"  # lambda_020 etc.
    print("Downloading frames for", tag)

    for frame in range(N_FRAMES):
        view.frame = frame
        sleep(0.6)

        view.download_image(filename=f"{tag}_frame_{frame:03d}.png")

        # throttle: give the browser time to complete the download
        sleep(1.0)

Output()

In [97]:
import glob, os
import moviepy as mpy

DOWNLOADS = os.path.expanduser("~/Downloads")
OUTDIR = "lambda_gifs2"
os.makedirs(OUTDIR, exist_ok=True)

FPS = 8

for lam in lambdas:
    tag = f"lambda_{int(round(lam*100)):03d}"
    files = sorted(glob.glob(os.path.join(DOWNLOADS, f"{tag}_frame_*.png")))

    if not files:
        print("no frames for", lam)
        continue

    clip = mpy.ImageSequenceClip(files, fps=FPS)
    out = os.path.join(OUTDIR, f"trajectory_{tag}.gif")
    clip.write_gif(out, fps=FPS)

    print("saved", out)

MoviePy - Building file lambda_gifs2/trajectory_lambda_000.gif with imageio.


saved lambda_gifs2/trajectory_lambda_000.gif
MoviePy - Building file lambda_gifs2/trajectory_lambda_020.gif with imageio.


saved lambda_gifs2/trajectory_lambda_020.gif
MoviePy - Building file lambda_gifs2/trajectory_lambda_040.gif with imageio.


saved lambda_gifs2/trajectory_lambda_040.gif
MoviePy - Building file lambda_gifs2/trajectory_lambda_060.gif with imageio.


saved lambda_gifs2/trajectory_lambda_060.gif
MoviePy - Building file lambda_gifs2/trajectory_lambda_080.gif with imageio.


saved lambda_gifs2/trajectory_lambda_080.gif
MoviePy - Building file lambda_gifs2/trajectory_lambda_100.gif with imageio.


saved lambda_gifs2/trajectory_lambda_100.gif


In [98]:
# ---- 5) Subsample + feed to MBAR ----
N_k = np.zeros(nstates, dtype=int)

for k in range(nstates):
    # detect equilibration on u_kk (self state)
    nequil, g, Neff_max = timeseries.detect_equilibration(u_kln[k, k, :])
    indices = timeseries.subsample_correlated_data(u_kln[k, k, nequil:], g=g)
    indices = indices + nequil

    N_k[k] = len(indices)
    # keep only uncorrelated samples, in-place
    u_kln[k, :, 0:N_k[k]] = u_kln[k, :, indices].T

# Trim trailing zeros if N_k < n_samples
n_max = np.max(N_k)
u_kln = u_kln[:, :, :n_max]

mbar = MBAR(u_kln, N_k)
result = mbar.compute_free_energy_differences()
DeltaF_ij   = result['Delta_f']
dDeltaF_ij  = result['dDelta_f']

# ---- 6) Report ΔF between λ=0 and λ=1 in kJ/mol ----
dF_dimless  = DeltaF_ij[0, -1]
ddF_dimless = dDeltaF_ij[0, -1]

dF_kJ  = dF_dimless  / beta
ddF_kJ = ddF_dimless / beta

print(f"ΔF(λ=0 → 1) = {dF_kJ:.3f} ± {ddF_kJ:.3f} kJ/mol")


******* JAX 64-bit mode is now on! *******
*     JAX is now set to 64-bit mode!     *
*   This MAY cause problems with other   *
*      uses of JAX in the same code.     *
******************************************



ΔF(λ=0 → 1) = 46.546 ± 4.976 kJ/mol


In [15]:

from openmmtools import alchemy, states, mcmc, multistate

# ---- 2) define ligand atoms for alchemy ----
# set this to your ligand residue name, e.g. "UNL" or whatever you used
LIGAND_RESNAME = "UNL"

ligand_atoms = [a.index for a in pdb.topology.atoms()
                if a.residue.name == LIGAND_RESNAME]
assert len(ligand_atoms) > 0, "No ligand atoms found; check LIGAND_RESNAME"

region = alchemy.AlchemicalRegion(alchemical_atoms=ligand_atoms)
factory = alchemy.AbsoluteAlchemicalFactory()
alchemical_system = factory.create_alchemical_system(system, region)

# ---- 3) thermodynamic states (lambda schedule) ----
lambdas = np.linspace(1.0, 0.0, 8)  # 8 windows from fully coupled → decoupled

thermo_states = []
for lam in lambdas:
    ts = states.ThermodynamicState(alchemical_system, temperature=300*unit.kelvin)
    ts.set_alchemical_parameters(
        lambda_sterics=lam,
        lambda_electrostatics=lam,
    )
    thermo_states.append(ts)

# ---- 4) sampler (REMD over lambdas) ----
sampler_state = states.SamplerState(positions=pdb.positions)

move = mcmc.LangevinDynamicsMove(
    timestep=2.0*unit.femtoseconds,
    collision_rate=1.0/unit.picosecond,
    n_steps=500,        # 500 steps per iteration
)

reporter = multistate.MultiStateReporter("complex.nc", checkpoint_interval=10)

sampler = multistate.ReplicaExchangeSampler(
    mcmc_moves=move,
    number_of_iterations=100,   # bump this up for real runs
)

sampler.create(thermo_states, sampler_state, reporter)
sampler.run()
reporter.close()

ModuleNotFoundError: No module named 'netCDF4'

In [ ]:
import numpy as np
from pymbar import MBAR

from openmmtools.multistate import MultiStateReporter

reporter = MultiStateReporter("complex.nc", open_mode="r")
nc = reporter._storage_netCDF  # internal handle; ok for quick script

u_kln = np.array(nc.variables["u_kln"][:])   # shape (K, K, N_max)
N_k   = np.array(nc.variables["N_k"][:])     # samples per state

reporter.close()

mbar = MBAR(u_kln, N_k)
res  = mbar.compute_free_energy_differences(compute_uncertainty=True)

Delta_f  = res["Delta_f"]      # in kT
dDelta_f = res["dDelta_f"]     # in kT
print(Delta_f)
print(dDelta_f)

In [ ]:
DeltaG_bind_kT = dG_complex_kT - dG_solvent_kT
DeltaG_bind_kcal = DeltaG_bind_kT * unit.AVOGADRO_CONSTANT_NA * unit.BOLTZMANN_CONSTANT_kB * 300*unit.kelvin
DeltaG_bind_kcal = DeltaG_bind_kcal.in_units_of(unit.kilocalories_per_mole).value
print("ΔG_bind (kcal/mol):", DeltaG_bind_kcal)

In [12]:
import os
from alchemtest.gmx import load_ABFE
from alchemlyb.workflows import ABFE

In [3]:
# Path to example GROMACS protein–ligand ABFE data
data = load_ABFE()
complex_dir = os.path.dirname(data["data"]["complex"][0])

# Set up ABFE workflow (GROMACS example, results in kcal/mol)
workflow = ABFE(
    T=298,
    units="kcal/mol",
    software="GROMACS",
    dir=complex_dir,
    prefix="dhdl",
    suffix="xvg",
    outdirectory=".",
)

# Run full analysis using MBAR only
workflow.run(
    skiptime=10,           # ps to discard as equilibration
    uncorr="dhdl",         # use dHdl for decorrelation (as in docs)
    threshold=50,          # min uncorrelated samples per state
    estimators=("MBAR",),  # only MBAR
    overlap="O_MBAR.pdf",  # overlap plot
    breakdown=True,
    forwrev=10,            # convergence plot
)

# workflow.summary is a DataFrame with per-window and per-stage ΔG
print("Full MBAR summary:")
print(workflow.summary)

# Extract total binding free energy (MBAR)
total_mbar = workflow.summary.xs("Stages").loc["TOTAL", ["MBAR", "MBAR_Error"]]
print("\nAbsolute binding free energy (MBAR):")
print(total_mbar)

2025-12-04 03:34:45.020 | INFO     | alchemlyb.workflows.base:__init__:57 - Alchemlyb Version: f2.5.0
2025-12-04 03:34:45.020 | INFO     | alchemlyb.workflows.base:__init__:58 - Set Temperature to 298 K.
2025-12-04 03:34:45.021 | INFO     | alchemlyb.workflows.base:__init__:60 - Set Software to GROMACS.
2025-12-04 03:34:45.021 | INFO     | alchemlyb.workflows.abfe:__init__:86 - Initialise Alchemlyb ABFE Workflow
2025-12-04 03:34:45.022 | INFO     | alchemlyb.workflows.abfe:update_units:339 - Set unit to kcal/mol.
2025-12-04 03:34:45.022 | INFO     | alchemlyb.workflows.abfe:__init__:88 - Finding files with prefix: dhdl, suffix: xvg under directory /Users/luismantilla/Desktop/UofT/NVIDIA/trovolut/.venv/lib/python3.13/site-packages/alchemtest/gmx/ABFE/complex produced by GROMACS
2025-12-04 03:34:45.023 | INFO     | alchemlyb.workflows.abfe:__init__:105 - Found 30 xvg files.
2025-12-04 03:34:45.024 | INFO     | alchemlyb.workflows.abfe:__init__:106 - Unsorted file list: 
 /Users/luismanti

Full MBAR summary:
                      MBAR  MBAR_Error
States 0 -- 1     0.041637    0.001108
       1 -- 2     0.056413    0.001143
       2 -- 3     0.082767    0.001297
       3 -- 4     0.073003    0.000953
       4 -- 5     0.065978    0.000770
       5 -- 6     0.219002    0.002184
       6 -- 7     0.252082    0.002186
       7 -- 8     0.201701    0.001646
       8 -- 9     0.271620    0.002180
       9 -- 10    0.221274    0.001833
       10 -- 11   2.207077    0.007394
       11 -- 12   1.788533    0.009023
       12 -- 13   1.379740    0.010210
       13 -- 14   0.950040    0.010457
       14 -- 15   0.574042    0.007762
       15 -- 16   0.566315    0.009368
       16 -- 17   1.106926    0.026699
       17 -- 18   1.129788    0.025191
       18 -- 19   1.243310    0.019638
       19 -- 20   1.392342    0.015664
       20 -- 21   1.583639    0.012221
       21 -- 22   0.855413    0.005844
       22 -- 23   0.884607    0.006112
       23 -- 24   0.895089    0.006831
      

In [3]:
# absolute_binding_free_energy.py

from openmm.app import PDBFile, Modeller, ForceField, Simulation
from openmm import unit, Context, Platform, LangevinIntegrator
# from openmmtools.alchemy import AbsoluteAlchemicalFactory, AlchemicalRegion, AlchemicalState

from openmmforcefields.generators import GAFFTemplateGenerator
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pymbar
import alchemlyb

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [ ]:

# 1. Load protein PDB
protein_path = "examples/qmmm/biocomplex/3GPH_chainA.pdb"
pdb = PDBFile(protein_path)  # loads topology and coordinates [oai_citation:4‡docs.openmm.org](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#:~:text=pdb%20%3D%20PDBFile%28%27input)

# 2. Load ligand SDF with RDKit and generate 3D conformer
ligand_sdf = "examples/qmmm/biocomplex/C2H6S2.sdf"
ligand = Chem.SDMolSupplier(ligand_sdf, removeHs=False)[0]
AllChem.EmbedMolecule(ligand, AllChem.ETKDG())


In [1]:
"""
Alchemical free energy (cookbook-style) for 3GPH_chainA + DMS (C2H6S2)
- Follows the OpenMM cookbook LJ tutorial pattern with CustomNonbondedForce + lambda
- Uses Amber14 + GAFF, implicit solvent (GBN2)
- Computes ΔG by MBAR from u_kln[k,l,n] like in the tutorial
"""

from openmm.app import PDBFile, Modeller, ForceField, Simulation
from openmm import (
    unit,
    LangevinIntegrator,
    NonbondedForce,
    CustomNonbondedForce,
)
from openff.toolkit.topology import Molecule as OFFMol
from rdkit import Chem
from rdkit.Chem import AllChem
from openmmforcefields.generators import GAFFTemplateGenerator
import numpy as np
from pymbar import MBAR, timeseries
from openmm.unit import AVOGADRO_CONSTANT_NA, BOLTZMANN_CONSTANT_kB

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [2]:
import sys 
sys.path.append("../../../")  # to import from parent dir

!pwd

/Users/luismantilla/Desktop/UofT/NVIDIA/trovolut/testing_scripts/notebooks/mbar


In [10]:

# ----------------------------------------------------------------------
# 1. Load protein + ligand complex
# ----------------------------------------------------------------------
# Option A: you prepared a merged PDB with protein + DMS already placed
# complex_pdb_file = "3GPH_DMS_complex.pdb"  # protein + ligand in binding site
# pdb = PDBFile(complex_pdb_file)

# Option B (commented): if you only have separate files, you can roughly place DMS
# and save the result as a new PDB later.
protein_pdb = PDBFile("../../../examples/qmmm/biocomplex/3GPH_chainA.pdb")
rdmol = Chem.SDMolSupplier("../../../examples/qmmm/biocomplex/C2H6S2.sdf", removeHs=False)[0]
AllChem.EmbedMolecule(rdmol, AllChem.ETKDG())
off_lig = OFFMol.from_rdkit(rdmol)
modeller = Modeller(protein_pdb.topology, protein_pdb.positions)
modeller.add(off_lig.to_topology().to_openmm(), off_lig.conformers[0].to_openmm())
PDBFile.writeFile(modeller.topology, modeller.positions, open("3GPH_DMS_complex.pdb","w"))

complex_pdb_file = "3GPH_DMS_complex.pdb"  # protein + ligand in binding site
pdb = PDBFile(complex_pdb_file)


In [11]:
from openmm.app import PDBFile, Modeller, ForceField, Simulation
from openmm import unit, LangevinIntegrator, NonbondedForce, CustomNonbondedForce
from openmmforcefields.generators import GAFFTemplateGenerator
from openff.toolkit.topology import Molecule as OFFMol
from rdkit import Chem
from rdkit.Chem import AllChem

complex_pdb_file = "3GPH_DMS_complex.pdb"
pdb = PDBFile(complex_pdb_file)

ligand_resname = "DMS"  # or "LIG" etc., make sure this matches your PDB
ligand_atoms = [a for a in pdb.topology.atoms() if a.residue.name == ligand_resname]
ligand_indices = [a.index for a in ligand_atoms]

# GAFF/OpenFF ligand
off_lig = OFFMol.from_smiles("CSSC")  # DMS

gaff = GAFFTemplateGenerator(molecules=[off_lig])
forcefield = ForceField("amber14-all.xml", "implicit/gbn2.xml")
forcefield.registerTemplateGenerator(gaff.generator)

# Modeller for protein+ligand
modeller = Modeller(pdb.topology, pdb.positions)

# 1) Add hydrogens WITHOUT passing the forcefield
modeller.addHydrogens()   # <--- change here

# 2) Create system with ignoreExternalBonds=True
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=NonbondedForce.NoCutoff,
    constraints=None,
    ignoreExternalBonds=True,   # <--- important
)

Did not recognize residue UNK; did you forget to call .add_molecules() to add it?


ValueError: No template found for residue 461 (UNK).  The set of atoms is similar to ACE, but is missing 1 O atom and has 3 H atoms and 2 S atoms too many.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template

In [9]:

# ----------------------------------------------------------------------
# 3. Build alchemical CustomNonbondedForce (softcore LJ) like cookbook
# ----------------------------------------------------------------------
# Get NonbondedForce
nbforce = None
for f in system.getForces():
    if isinstance(f, NonbondedForce):
        nbforce = f
        break
if nbforce is None:
    raise RuntimeError("No NonbondedForce found in system.")

n_particles = system.getNumParticles()
# Assume ligand atoms are contiguous and last in the topology
# (if not, we use the indices from the residue selection above)
ligand_set = set(ligand_indices)
other_set = set(range(n_particles)) - ligand_set

# Define softcore LJ energy (cookbook style) with global parameter lambda
energy_function = (
    "lambda_alch*4*epsilon*x*(x-1.0);"
    "x = (sigma/reff_sterics)^6;"
    "reff_sterics = sigma*(0.5*(1.0-lambda_alch) + (r/sigma)^6)^(1/6);"
    "sigma = 0.5*(sigma1+sigma2);"
    "epsilon = sqrt(epsilon1*epsilon2);"
)

custom_force = CustomNonbondedForce(energy_function)
custom_force.setNonbondedMethod(CustomNonbondedForce.NoCutoff)

# Global lambda parameter
custom_force.addGlobalParameter("lambda_alch", 1.0)

# Per-particle parameters (sigma, epsilon)
custom_force.addPerParticleParameter("sigma")
custom_force.addPerParticleParameter("epsilon")

# Copy LJ parameters from NonbondedForce, and zero out LJ on ligand in the original NonbondedForce
for index in range(n_particles):
    charge, sigma, epsilon = nbforce.getParticleParameters(index)
    custom_force.addParticle([sigma, epsilon])
    if index in ligand_set:
        # remove LJ for ligand in original force; handled by custom force now
        nbforce.setParticleParameters(index, charge, sigma, 0.0 * epsilon)

# Only interactions between ligand atoms and everything else are alchemical
custom_force.addInteractionGroup(ligand_set, other_set)
system.addForce(custom_force)

# ----------------------------------------------------------------------
# 4. Simulation setup
# ----------------------------------------------------------------------
temperature = 300 * unit.kelvin
collision_rate = 1.0 / unit.picosecond
timestep = 2.0 * unit.femtoseconds

integrator = LangevinIntegrator(temperature, collision_rate, timestep)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

print("Minimizing...")
simulation.minimizeEnergy()


NameError: name 'system' is not defined

In [ ]:

# ----------------------------------------------------------------------
# 5. Alchemical λ schedule and MBAR-style data collection (u_kln)
# ----------------------------------------------------------------------
nsteps = 500     # steps between samples (small for demo; increase in real runs)
niterations = 200  # samples per state (increase later)
lambdas = np.linspace(1.0, 0.0, 8)  # 8 alchemical states from fully on → off
nstates = len(lambdas)

u_kln = np.zeros((nstates, nstates, niterations), dtype=np.float64)

kT = (AVOGADRO_CONSTANT_NA * BOLTZMANN_CONSTANT_kB * temperature).value_in_unit(
    unit.kilojoule_per_mole
)

for k in range(nstates):
    print(f"=== Sampling state k={k}, lambda={lambdas[k]:.3f} ===")
    for n in range(niterations):
        # Set simulation to state k
        simulation.context.setParameter("lambda_alch", float(lambdas[k]))
        simulation.step(nsteps)
        # Compute reduced potentials at all lambda states (l)
        for l in range(nstates):
            simulation.context.setParameter("lambda_alch", float(lambdas[l]))
            E = simulation.context.getState(getEnergy=True).getPotentialEnergy()
            u_kln[k, l, n] = E.value_in_unit(unit.kilojoule_per_mole) / kT

# ----------------------------------------------------------------------
# 6. Analyze with MBAR (same structure as cookbook tutorial)
# ----------------------------------------------------------------------
N_k = np.zeros(nstates, dtype=np.int32)

for k in range(nstates):
    # detect equilibration using diagonal (k,k) timeseries
    (nequil, g, Neff_max) = timeseries.detect_equilibration(u_kln[k, k, :])
    indices = timeseries.subsample_correlated_data(u_kln[k, k, :], g=g)
    N_k[k] = len(indices)
    u_kln[k, :, 0:N_k[k]] = u_kln[k, :, indices].T

# Trim unused samples (optional)
max_N = N_k.max()
u_kln = u_kln[:, :, :max_N]

mbar = MBAR(u_kln, N_k)
results = mbar.compute_free_energy_differences(compute_uncertainty=True)
Delta_f = results["Delta_f"]
dDelta_f = results["dDelta_f"]

# ΔG from λ=1 (fully interacting) to λ=0 (decoupled)
dg = Delta_f[nstates - 1, 0]
ddg = dDelta_f[nstates - 1, 0]

print(f"\nEstimated ΔG(alchemical LJ decoupling of DMS) = {dg:.3f} ± {ddg:.3f} kT")
print("To convert to kJ/mol: multiply by kT; to kcal/mol: multiply by kT/4.184")